<a href="https://colab.research.google.com/github/jnahMoch/revisedBookRecommender/blob/main/vector_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
!pip install langchain langchain-community langchain-openai langchain-chroma chromadb pandas python-dotenv

In [49]:
import os
import pandas as pd
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# ===================================================
# 1. SAFELY FETCH YOUR API KEY FROM COLAB SECRETS
# ===================================================
try:
    # This fetches the hidden key without printing it on the screen
    # Explicitly handle potential duplicated keys or internal newlines
    MY_API_KEY = userdata.get('OPENAI_API_KEY').split('\n')[0].strip()
    os.environ["OPENAI_API_KEY"] = MY_API_KEY
    print("✅ Securely fetched OpenAI API Key from Colab Secrets!")
except Exception as e:
    raise ValueError("Could not find 'OPENAI_API_KEY' in your Colab Secrets. Please check Step 1!")

✅ Securely fetched OpenAI API Key from Colab Secrets!


In [50]:
# ===================================================
# 2. LOAD & CLEAN DATASET
# ===================================================
print("🔄 Loading and cleaning dataset...")
books = pd.read_csv("books_with_categories_and_genre.csv")

# Fill missing items so string combining doesn't crash
books['title'] = books['title'].fillna('')
books['genre'] = books['genre'].fillna('')
books['simple_categories_binary'] = books['simple_categories_binary'].fillna('')
books['description'] = books['description'].fillna('')

# Build the custom tagged description where ISBN13 is always the FIRST word
books["tagged_description"] = (
    books["isbn13"].astype(str) + " " +
    books["title"] + " " +
    books["genre"] + " " +
    books["simple_categories_binary"] + " " +
    books["description"]
)

# Strip out empty strings
books = books[books["tagged_description"].str.strip() != ""]

# Save out to flat text loader file
books["tagged_description"].to_csv("tagged_description.txt", sep="\t", index=False, header=False)
print("   -> Created 'tagged_description.txt' successfully.")


🔄 Loading and cleaning dataset...
   -> Created 'tagged_description.txt' successfully.


In [51]:
# ===================================================
# 3. PARSE DOCUMENTS LINE-BY-LINE
# ===================================================
raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(
    chunk_size=500, # Increased chunk size to reduce warnings
    chunk_overlap=0,
    separator="\n",
    strip_whitespace=True
)
documents = text_splitter.split_documents(raw_documents)
print(f"   -> Split texts into {len(documents)} distinct book blocks.")

   -> Split texts into 908 distinct book blocks.


In [52]:
# ===================================================
# 4. INITIALIZE CHROMA WITH VAULTED API KEY
# ===================================================
print("Initialized Vector Database")

embedding_model = OpenAIEmbeddings(api_key=MY_API_KEY)

db_books = Chroma.from_documents(
    documents,
    embedding=embedding_model
)

Initialized Vector Database


In [53]:
# ===================================================
# 5. CRASH-PROOF RECOMMENDATION FUNCTION
# ===================================================
def retrieve_semantic_recommendations(query: str, top_k: int = 10) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)
    books_list = []

    for i in range(len(recs)):
        try:
            content = recs[i].page_content.strip('"').strip()
            if not content:
                continue

            words = content.split()
            if len(words) > 0:
                raw_isbn = words[0].strip()
                # Clean datatype parsing to match integers
                clean_isbn = int(float(raw_isbn))
                books_list.append(clean_isbn)
        except (ValueError, IndexError):
            continue

    # Return matching results safely
    matched_books = books[books["isbn13"].isin(books_list)]
    return matched_books.head(top_k)

In [56]:
# ===================================================
# 6. RUN A TEST QUERY AUTOMATICALLY
# ===================================================
print("\n--- SAMPLE SEARCH RESULTS ---")
results = retrieve_semantic_recommendations("A suspenseful fiction murder mystery book", top_k=5)
display(results[['title','authors', 'genre', 'simple_categories_binary', 'average_rating']])



--- SAMPLE SEARCH RESULTS ---


,title,authors,genre,simple_categories_binary,average_rating
13,Murder in LaMut,Raymond E. Feist;Joel Rosenberg,Literature & Drama,fiction,3.70
25,Murder in Mesopotamia,Agatha Christie,Literature & Drama,fiction,3.89
28,Appointment with Death,Agatha Christie,Literature & Drama,fiction,3.86
29,Hallowe'en Party,Agatha Christie,Literature & Drama,fiction,3.66
34,A Murder is Announced,Agatha Christie,Literature & Drama,fiction,3.98


In [55]:
books.head()

,isbn13,isbn10,title,authors,categories,description,thumbnail,average_rating,num_pages,cleaned_description,simple_categories_binary,genre,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,http://books.google.com/books/content?id=KQZCP...,3.85,247.0,novel reader critic eagerly anticipating decad...,fiction,Literature & Drama,9780002005883 Gilead Literature & Drama fictio...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Fiction,A new 'Christie for Christmas' -- a full-lengt...,http://books.google.com/books/content?id=gA5GP...,3.83,241.0,new christie christmas fulllength novel adapte...,fiction,Literature & Drama,9780002261982 Spider's Web Literature & Drama ...
2,9780006163831,0006163831,The One Tree,Stephen R. Donaldson,Fiction,Volume Two of Stephen Donaldson's acclaimed se...,http://books.google.com/books/content?id=OmQaw...,3.97,479.0,volume two stephen donaldsons acclaimed second...,fiction,Literature & Drama,9780006163831 The One Tree Literature & Drama ...
3,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",http://books.google.com/books/content?id=FKo2T...,3.93,512.0,memorable mesmerizing heroine jennifer brillia...,fiction,Literature & Drama,9780006178736 Rage of angels Literature & Dram...
4,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Religion,Lewis' work on the nature of love divides love...,http://books.google.com/books/content?id=XhQ5X...,4.15,170.0,lewis work nature love divide love four catego...,nonfiction,Philosophy & Religion,9780006280897 The Four Loves Philosophy & Reli...


In [58]:
#Test here:

print("What kind of book are looking for?")
search_query = input("Type here: ")
results =  retrieve_semantic_recommendations(search_query, top_k=5)
display(results[['title','authors', 'genre', 'simple_categories_binary', 'average_rating', 'description']])

What kind of book are looking for?
Type here: coming of age


,title,authors,genre,simple_categories_binary,average_rating,description
14,Mystical Paths,Susan Howatch,Literature & Drama,fiction,4.23,1968 finds Nicholas Darrow wrestling with pers...
20,Girls' Night in,Jessica Adams;Chris Manby;Fiona Walker,Literature & Drama,fiction,3.26,'Girls' Night In' features stories about growi...
216,Missing Mom,Joyce Carol Oates,Literature & Drama,fiction,3.54,"Nikki Eaton, single, thirty-one, sexually libe..."
221,Black Boy,Richard Wright,Biography & History,nonfiction,4.05,Traces the author's coming of age in the Jim C...
240,How to Be Popular,Meg Cabot,Childrens Books,fiction,3.54,Sixteen-year-old Steph Landry finds an old boo...
